# Import Libraries and Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from math import erfc, sqrt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
path = '/content/drive/MyDrive/ASKISI1_DATA_2026.xlsx' #'/content/ASKISI1_DATA_2026.xlsx'
df = pd.read_excel(path)

# Question 1 — Exploratory Data Analysis

In [ ]:
# Scatter plot of propeller shaft power over time (weeks)
plt.figure(figsize=(15, 7))
sns.scatterplot(x=df['TIME'], y=df['PROPELLER SHAFT POWER  [kW]'], alpha=0.6, s=10)

plt.title('Scatter Plot of PROPELLER SHAFT POWER [kW] over Time (in weeks)')
plt.ylabel('PROPELLER SHAFT POWER [kW]')

plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.gca().xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.xticks(rotation=45, ha='right')

plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot of mean draft versus trim
plt.figure(figsize=(15, 7))
sns.scatterplot(x=df['TRIM [m]'], y=df['Tmean [m]'], alpha=0.6, s=10, label='Data Points')

plt.scatter(y=6.5, x=3, color='b', marker='o', label='Tballast')
plt.scatter(y=12.2, x=0,  color='g', marker='o', label='Tdesign')
plt.scatter(y=14.4, x=0, color='r', marker='o', label='Tscantling')

plt.title('Scatter Plot of Tmean [m] vs TRIM [m]')
plt.xlabel('TRIM [m]')
plt.ylabel('Tmean [m]')

plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot of speed over ground over time (weeks)
df['TIME'] = pd.to_datetime(df['TIME'])

plt.figure(figsize=(15, 7))
sns.scatterplot(x=df['TIME'], y=df['SOG [kn]'], alpha=0.6, s=10)

plt.title('Scatter Plot of SOG [kn] over Time (in weeks)')
plt.xlabel('Time (Weeks)')
plt.ylabel('Speed Over Ground (knots)')


plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.gca().xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.xticks(rotation=45, ha='right')

plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot of speed through water over time (weeks)
df['TIME'] = pd.to_datetime(df['TIME'])

plt.figure(figsize=(15, 7))
sns.scatterplot(x=df['TIME'], y=df['STW [kn]'], alpha=0.6, s=10)

plt.title('Scatter Plot of STW [kn] over Time (in weeks)')
plt.xlabel('Time (Weeks)')
plt.ylabel('Speed Through Water (knots)')

plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.gca().xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.xticks(rotation=45, ha='right')

plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot of mean draft over time (weeks)

plt.figure(figsize=(15, 7))
sns.scatterplot(x=df['TIME'], y=df['Tmean [m]'], alpha=0.6, s=10, label='Tmean [m] Data')
plt.title('Scatter Plot of Tmean over Time (in weeks)')
plt.xlabel('Time (Weeks)')
plt.ylabel('Tmean (m)')

plt.axhline(y=6.5, color='b', linestyle='-', label='Tballast')
plt.axhline(y=12.2, color='g', linestyle='-', label='Tdesign')
plt.axhline(y=14.4, color='r', linestyle='-', label='Tscantling')


plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.gca().xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.xticks(rotation=45, ha='right')

plt.grid(True, linestyle='-', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot of propeller revolutions over time (weeks)
plt.figure(figsize=(15, 7))
sns.scatterplot(x=df['TIME'], y=df['PROPELLER REV [rpm]'], alpha=0.6, s=10)

plt.title('Scatter Plot of PROPELLER REV [rpm] over Time (in weeks)')
plt.xlabel('Time (Weeks)')
plt.ylabel('PROPELLER REV [rpm]')

plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.gca().xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Histogram of propeller revolutions
plt.figure(figsize=(10, 6))
sns.histplot(df['PROPELLER REV [rpm]'], bins=85)

plt.title('Histogram of PROPELLER REV [rpm]')
plt.xlabel('PROPELLER REV [rpm]')
plt.ylabel('Frequency')

plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


# Question 2 — Wind Resistance Correction and Ship Performance

In [ ]:
# Standardize column names
df.columns = df.columns.str.strip().str.replace(r'\s+', ' ', regex=True)

# Filter the data
df_q2 = df.copy()

mask = (
    (df_q2['ME FOC [t/day]'] > 0) &
    (df_q2['STW [kn]'] > 0) &
    (df_q2['SOG [kn]'] > 0) &
    (df_q2['RELATIVE WIND SPEED [m/s]'] <= 18)
)

df_q2 = df_q2[mask].copy()
len(df_q2)

In [ ]:
# Histogram of main engine fuel oil consumption [t/day]
plt.figure(figsize=(10, 6))
sns.histplot(df_q2['ME FOC [t/day]'], bins=34)
plt.title('Histogram of ME FOC [t/day]')
plt.xlabel('ME FOC [t/day]')
plt.ylabel('Frequency')

plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
KNOT_TO_MS = 0.514444
# Values specified in the assignment
T_ref = (8.0 + 5.0) / 2.0      #ballast
B_ship = 32.26
A_ref = 822.0
Za_ref = 38.0
Zref_ref = 10.0

In [ ]:
# Known quantities
# v_g: ship speed relative to the ground
df_q2['vg [m/s]'] = df_q2['SOG [kn]'] * KNOT_TO_MS

# v_wr: relative wind speed
df_q2['vwr [m/s]'] = df_q2['RELATIVE WIND SPEED [m/s]']

# Wind direction and ship heading
df_q2['ψ_wr [deg]'] = df_q2['RELATIVE WIND DIRECTION [deg]']
df_q2['ψ_o [deg]'] = df_q2['SHIP HEADING [deg]']

psi_wr = np.deg2rad(df_q2['ψ_wr [deg]'].to_numpy())
psi_o  = np.deg2rad(df_q2['ψ_o [deg]'].to_numpy())

vg  = df_q2['vg [m/s]'].to_numpy()
vwr = df_q2['vwr [m/s]'].to_numpy()

In [ ]:
# v_wt
vwt = np.sqrt(vwr**2 + vg**2 - 2.0 * vwr * vg * np.cos(psi_wr))

# psi_wt
y_wt = vwr * np.sin(psi_wr + psi_o) - vg * np.sin(psi_o)
x_wt = vwr * np.cos(psi_wr + psi_o) - vg * np.cos(psi_o)

psi_wt = np.mod(np.arctan2(y_wt, x_wt), 2*np.pi)

df_q2['vwt [m/s]'] = vwt
df_q2['ψ_wt [rad]'] = psi_wt
df_q2['ψ_wt [deg]'] = np.rad2deg(psi_wt)

In [ ]:
# Reference heights and projected windage area
T_actual = df_q2['Tmean [m]'].to_numpy()

# ΔT = T_ref - T
delta_T = T_ref - T_actual

# A = A_ref + ΔT * B
A_actual = A_ref + delta_T * B_ship

# Z_a = Z_a,ref + ΔT
Za_actual = Za_ref + delta_T

# Z_ref = [A_ref*(Z_ref,ref + ΔT) + 0.5*B*ΔT^2] / A
Zref_actual = (A_ref * (Zref_ref + delta_T) + 0.5 * B_ship * delta_T**2) / A_actual

df_q2['delta_T [m]'] = delta_T
df_q2['A [m2]'] = A_actual
df_q2['Za [m]'] = Za_actual
df_q2['Zref [m]'] = Zref_actual

In [ ]:
# True wind speed at the reference height
vwt_ref = vwt * (Zref_actual / Za_actual) ** (1.0 / 7.0)

df_q2['vwt_ref [m/s]'] = vwt_ref

In [ ]:
# Relative wind speed at the reference height
# v_wr,ref
vwr_ref = np.sqrt(vwt_ref**2 + vg**2 + 2.0 * vwt_ref * vg * np.cos(psi_wt - psi_o))

# ψ_wr,ref
y_wr_ref = vwt_ref * np.sin(psi_wt - psi_o)
x_wr_ref = vg + vwt_ref * np.cos(psi_wt - psi_o)

psi_wr_ref = np.mod(np.arctan2(y_wr_ref, x_wr_ref), 2*np.pi)

df_q2['vwr_ref [m/s]'] = vwr_ref
df_q2['ψ_wr_ref [rad]'] = psi_wr_ref
df_q2['ψ_wr_ref [deg]'] = np.rad2deg(psi_wr_ref)

In [ ]:
# Quick sanity check of calculated values
cols_check = [
    'TIME',
    'SOG [kn]',
    'STW [kn]',
    'SHIP HEADING [deg]',
    'RELATIVE WIND DIRECTION [deg]',
    'RELATIVE WIND SPEED [m/s]',
    'vg [m/s]',
    'vwr [m/s]',
    'vwt [m/s]',
    'ψ_wt [deg]',
    'delta_T [m]',
    'A [m2]',
    'Za [m]',
    'Zref [m]',
    'vwt_ref [m/s]',
    'vwr_ref [m/s]',
    'ψ_wr_ref [deg]'
]

print(df_q2[cols_check].head(10))

In [ ]:
angles_crw = np.array([
    0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165, 180,
    195, 210, 225, 240, 255, 270, 285, 300, 315, 330, 345, 360
], dtype=float)
# Wind resistance coefficient values
cwr_table = np.array([
    0.72, 0.60, 0.50, 0.43, 0.25, 0.17, -0.10, -0.20, -0.25, -0.40, -0.50, -0.52, -0.53,
    -0.52, -0.50, -0.40, -0.25, -0.20, -0.10, 0.17, 0.25, 0.43, 0.50, 0.60, 0.72
], dtype=float)
# Linear interpolation
df_q2['Cwr'] = np.interp(df_q2['ψ_wr_ref [deg]'], angles_crw, cwr_table)

In [ ]:
# Rrw
rho_a = 1.225

df_q2['Rrw [N]'] = 0.5 * rho_a * (df_q2['vwr_ref [m/s]']**2) * df_q2['A [m2]'] * df_q2['Cwr']

print(df_q2[['TIME', 'vwr_ref [m/s]', 'A [m2]', 'Cwr', 'Rrw [N]']].head(10))

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(df_q2['ψ_wr_ref [deg]'], df_q2['Rrw [N]'], s=8, alpha=0.5)
plt.xlabel('ψ_wr_ref [deg]')
plt.ylabel('Rrw [N]')
plt.title('Wind Resistance Rrw vs Relative Wind Direction')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# R0w
rho_a = 1.225
C0w = 0.72   # Wind resistance coefficient at 0 degrees

df_q2['R0w [N]'] = 0.5 * rho_a * (df_q2['vg [m/s]']**2) * df_q2['A [m2]'] * C0w

print(df_q2[['TIME', 'vg [m/s]', 'A [m2]', 'R0w [N]']].head(10))

In [ ]:
# Wind power correction and corrected shaft power
eta_D0 = 0.7
eta_DM = 0.7

df_q2['Delta_Pw [W]'] = ((df_q2['Rrw [N]'] - df_q2['R0w [N]']) * df_q2['vg [m/s]']) / eta_D0

# Convert to kW
df_q2['Delta_Pw [kW]'] = df_q2['Delta_Pw [W]'] / 1000.0

# Corrected shaft power
df_q2['PD_corr [kW]'] = df_q2['PROPELLER SHAFT POWER [kW]'] - df_q2['Delta_Pw [kW]']

print(df_q2[['TIME',
             'PROPELLER SHAFT POWER [kW]',
             'Rrw [N]',
             'R0w [N]',
             'Delta_Pw [kW]',
             'PD_corr [kW]']].head(10))

In [ ]:
plt.figure(figsize=(15,7))
plt.scatter(df_q2['TIME'], df_q2['PROPELLER SHAFT POWER [kW]'], s=8, alpha=0.5, label='Measured Power')
plt.scatter(df_q2['TIME'], df_q2['PD_corr [kW]'], s=8, alpha=0.5, label='Corrected Power')

plt.xlabel('Time')
plt.ylabel('Power [kW]')
plt.title('Measured and Wind-Corrected Shaft Power over Time')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Select a short time window (one day)
start_time = '2021-07-15 00:00:00'
end_time   = '2021-07-15 23:59:59'

df_short = df_q2[
    (df_q2['TIME'] >= start_time) &
    (df_q2['TIME'] <= end_time)
].copy()

print("Number of data points:", len(df_short))
print(df_short['TIME'].min(), df_short['TIME'].max())

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(df_short['TIME'], df_short['PROPELLER SHAFT POWER [kW]'],
         label='Measured Power', color='blue')
plt.plot(df_short['TIME'], df_short['PD_corr [kW]'],
         label='Corrected Power', color='red')

plt.xlabel('Time')
plt.ylabel('Power [kW]')
plt.title('Measured vs Corrected Power (short time interval)')
plt.legend()
plt.grid(True)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(df_short['TIME'], df_short['vwr_ref [m/s]'],
         color='green')

plt.xlabel('Time')
plt.ylabel('Relative wind speed [m/s]')
plt.title('Relative Wind Speed (short time interval)')
plt.grid(True)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(df_short['TIME'], df_short['ψ_wr_ref [deg]'],
         color='orange')

plt.xlabel('Time')
plt.ylabel('Relative wind direction [deg]')
plt.title('Relative Wind Direction (short time interval)')
plt.grid(True)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Points defining the reference speed-power curves
ref_conditions = {
    'ballast': {
        'T_ref': 6.5,
        'cb_ref': 0.807,
        'Delta_ref': 37166.0,
        'V': np.array([10.20, 11.50, 12.28, 13.50, 14.47, 14.84, 15.00, 15.27, 15.50, 15.75]),
        'P': np.array([1891.82, 2728.75, 3441.24, 4915.35, 6389.46, 7038.90, 7373.52, 7864.56, 8355.60, 8847.63]),
    },
    'design': {
        'T_ref': 12.2,
        'cb_ref': 0.864,
        'Delta_ref': 78711.0,
        'V': np.array([8.74, 10.05, 11.01, 12.60, 13.70, 14.20, 14.38, 14.60, 14.80, 15.00, 15.40]),
        'P': np.array([1891.82, 2728.75, 3441.24, 4915.35, 6389.46, 7038.90, 7373.52, 7864.56, 8355.60, 8847.63, 9830.70]),
    },
    'scantling': {
        'T_ref': 14.4,
        'cb_ref': 0.879,
        'Delta_ref': 92387.6,
        'V': np.array([7.63, 9.03, 9.99, 11.90, 13.10, 13.45, 13.67, 13.90, 14.10, 14.40, 14.80]),
        'P': np.array([1891.82, 2728.75, 3441.24, 4915.35, 6389.46, 7038.90, 7373.52, 7864.56, 8355.60, 8847.63, 9830.70]),
    }
}

# Linear regression in log space for the propeller-law power curve
for cond, vals in ref_conditions.items():
    logV = np.log(vals['V'])
    logP = np.log(vals['P'])
    b, loga = np.polyfit(logV, logP, 1)
    a = np.exp(loga)
    vals['a'] = a
    vals['b'] = b

In [ ]:
def choose_reference_condition(t_mean):
    drafts = {k: v['T_ref'] for k, v in ref_conditions.items()}
    return min(drafts, key=lambda k: abs(t_mean - drafts[k]))


def cb_actual(T_act, T_ref, cb_ref):
    return 1 - (1 - cb_ref) * (T_ref / T_act)**(1/3)


def displacement_actual(T_act, T_ref, cb_ref, Delta_ref):
    cb_act = cb_actual(T_act, T_ref, cb_ref)
    Delta_act = Delta_ref * (cb_act / cb_ref) * (T_act / T_ref)
    return Delta_act, cb_act


def speed_from_power(P, a, b):
    return (P / a)**(1/b)


def power_from_speed(V, a, b):
    return a * (V**b)

In [ ]:
records = []

for idx, row in df_q2.iterrows():

    T_act = row['Tmean [m]']
    Vm = row['STW [kn]']
    Pm = row['PD_corr [kW]']

    # Optionally skip observations that are likely to represent transient operating conditions
    #if Vm <= 5:
    #  continue

    cond = choose_reference_condition(T_act)
    ref = ref_conditions[cond]

    T_ref = ref['T_ref']
    cb_ref = ref['cb_ref']
    Delta_ref = ref['Delta_ref']
    a = ref['a']
    b = ref['b']

    Delta_act, cb_act = displacement_actual(T_act, T_ref, cb_ref, Delta_ref)

    if Delta_act <= 0 or Delta_ref <= 0:
        continue

    # Displacement deviation criterion
    disp_dev = abs(100 * (Delta_ref - Delta_act) / Delta_act)

    if disp_dev >= 5:
        continue

    # Ve
    Ve_ref = speed_from_power(Pm, a, b)
    Ve = Ve_ref * ((Delta_ref**(2/3)) / (Delta_act**(2/3)))**(1/3)

    # Pe
    Pe_ref = power_from_speed(Vm, a, b)
    Pe = Pe_ref * (Delta_act / Delta_ref)**(2/3)

    # Exclude small values to avoid poorly conditioned ratios
    if Pe < 1 or Ve < 0.1:
        continue

    Vd = 100 * (Vm - Ve) / Ve
    Pd = 100 * (Pm - Pe) / Pe

    records.append({
        'TIME': row['TIME'],
        'cond': cond,
        'Vm': Vm,
        'Pm': Pm,
        'Ve': Ve,
        'Pe': Pe,
        'Speed loss [%]': Vd,
        'Power deviation [%]': Pd
    })

df_pv = pd.DataFrame(records)

In [ ]:
# Create a DataFrame with the ship performance indicators
df_pv = pd.DataFrame(records)

# outliers
df_pv_filtered = df_pv[
    (df_pv['Speed loss [%]'] > -30) &
    (df_pv['Speed loss [%]'] < 10) &
    (df_pv['Power deviation [%]'] > -10) &
    (df_pv['Power deviation [%]'] < 110)
].copy()

# Summary statistics
print("Total valid points BEFORE filtering:", len(df_pv))
print("Total valid points AFTER filtering:", len(df_pv_filtered))

print("\nSpeed loss stats:")
print(df_pv_filtered['Speed loss [%]'].describe())

print("\nPower deviation stats:")
print(df_pv_filtered['Power deviation [%]'].describe())

In [ ]:
df_pv_filtered = df_pv_filtered.sort_values('TIME')

plt.figure(figsize=(15,7))
plt.scatter(df_pv_filtered['TIME'], df_pv_filtered['Speed loss [%]'], s=8)
plt.title('Speed Loss over Time')
plt.xlabel('Time')
plt.ylabel('Speed loss [%]')
plt.grid(True)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45)
plt.show()


plt.figure(figsize=(15,7))
plt.scatter(df_pv_filtered['TIME'], df_pv_filtered['Power deviation [%]'], s=8)
plt.title('Power Deviation over Time')
plt.xlabel('Time')
plt.ylabel('Power deviation [%]')
plt.grid(True)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45)
plt.show()

In [ ]:

# Prepare the data
df_plot = df_pv_filtered.copy()
df_plot['TIME'] = pd.to_datetime(df_plot['TIME'])
df_plot = df_plot.sort_values('TIME')

print(df_plot[['TIME', 'Speed loss [%]', 'Power deviation [%]']].head())

# Dry-docking period
drydock_start = pd.Timestamp('2021-05-18')
drydock_end   = pd.Timestamp('2021-06-14')

# Before dry-docking
df_pre = df_plot[df_plot['TIME'] < drydock_start].copy()

# After dry-docking
df_post = df_plot[df_plot['TIME'] > drydock_end].copy()

print('Pre-dry dock points :', len(df_pre))
print('Post-dry dock points:', len(df_post))

In [ ]:

# Statistical summary before and after dry-docking
summary = pd.DataFrame({
    'Period': ['Pre dry-dock', 'Post dry-dock'],
    'N points': [len(df_pre), len(df_post)],

    'Speed loss mean [%]': [
        df_pre['Speed loss [%]'].mean(),
        df_post['Speed loss [%]'].mean()
    ],
    'Speed loss median [%]': [
        df_pre['Speed loss [%]'].median(),
        df_post['Speed loss [%]'].median()
    ],
    'Speed loss std [%]': [
        df_pre['Speed loss [%]'].std(),
        df_post['Speed loss [%]'].std()
    ],

    'Power deviation mean [%]': [
        df_pre['Power deviation [%]'].mean(),
        df_post['Power deviation [%]'].mean()
    ],
    'Power deviation median [%]': [
        df_pre['Power deviation [%]'].median(),
        df_post['Power deviation [%]'].median()
    ],
    'Power deviation std [%]': [
        df_pre['Power deviation [%]'].std(),
        df_post['Power deviation [%]'].std()
    ]
})

print(summary)

In [ ]:
plt.figure(figsize=(15,6))

# All data points
plt.scatter(
    df_plot['TIME'],
    df_plot['Speed loss [%]'],
    s=8,
    alpha=0.25,
    label='Data points'
)

# Mean value before dry-docking
plt.hlines(
    y=df_pre['Speed loss [%]'].mean(),
    xmin=df_pre['TIME'].min(),
    xmax=df_pre['TIME'].max(),
    color='black',
    linewidth=2.5,
    label='Mean before dry-dock'
)

# Mean value after dry-docking
plt.hlines(
    y=df_post['Speed loss [%]'].mean(),
    xmin=df_post['TIME'].min(),
    xmax=df_post['TIME'].max(),
    color='black',
    linewidth=2.5,
    label='Mean after dry-dock'
)

# Vertical lines indicating the dry-docking period
plt.axvline(drydock_start, linestyle='--')
plt.axvline(drydock_end, linestyle='--')

plt.xlabel('Time')
plt.ylabel('Speed loss [%]')
plt.title('Speed Loss before and after Dry Dock')
plt.grid(True)
plt.legend(loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15,6))

# All data points
plt.scatter(
    df_plot['TIME'],
    df_plot['Power deviation [%]'],
    s=8,
    alpha=0.25,
    label='Data points'
)

# Mean value before dry-docking
plt.hlines(
    y=df_pre['Power deviation [%]'].mean(),
    xmin=df_pre['TIME'].min(),
    xmax=df_pre['TIME'].max(),
    color='black',
    linewidth=2.5,
    label='Mean before dry-dock'
)

# Mean value after dry-docking
plt.hlines(
    y=df_post['Power deviation [%]'].mean(),
    xmin=df_post['TIME'].min(),
    xmax=df_post['TIME'].max(),
    color='black',
    linewidth=2.5,
    label='Mean after dry-dock'
)

# Vertical lines indicating the dry-docking period
plt.axvline(drydock_start, linestyle='--')
plt.axvline(drydock_end, linestyle='--')

plt.xlabel('Time')
plt.ylabel('Power deviation [%]')
plt.title('Power Deviation before and after Dry Dock')
plt.grid(True)
plt.legend(loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,7))

for cond, vals in ref_conditions.items():
    V_plot = np.linspace(min(vals['V']), max(vals['V']), 100)
    P_plot = vals['a'] * V_plot**vals['b']
    plt.plot(V_plot, P_plot, label=cond)

plt.scatter(df_pv['Vm'], df_pv['Pm'], s=5, alpha=0.3, label='data')

plt.xlabel('Speed [kn]')
plt.ylabel('Power [kW]')
plt.title('Power-Speed Curves')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

# Corrected measurements
plt.scatter(df_q2['STW [kn]'], df_q2['PD_corr [kW]'],
            s=5, alpha=0.3, label='Corrected data')

# Reference curves
for cond, vals in ref_conditions.items():
    V_plot = np.linspace(min(vals['V']), max(vals['V']), 100)
    P_plot = vals['a'] * V_plot**vals['b']
    plt.plot(V_plot, P_plot, label=cond)

plt.xlabel('Speed [kn]')
plt.ylabel('Power [kW]')
plt.title('Power-Speed Diagram with Reference Curves')
plt.legend()
plt.grid(True)
plt.show()

# Question 3 — Engine Fuel Consumption and Power Estimation

In [ ]:

# Specific fuel oil consumption curve from engine shop tests
specific_fuel_consumption_curve = {
    'Pb': np.array([0.25*9930, 0.35*9930, 0.5*9930, 0.716*9930, 0.75*9930, 9930]),
    'Specific fuel consumption (g/kWh)': np.array([175.9, 172.8, 167.7, 166, 167.7, 172.6])}
df_specific_fuel_consumption_curve = pd.DataFrame(specific_fuel_consumption_curve)

plt.figure(figsize=(10, 6))
plt.scatter(df_specific_fuel_consumption_curve['Pb'], df_specific_fuel_consumption_curve['Specific fuel consumption (g/kWh)'])
plt.plot(df_specific_fuel_consumption_curve['Pb'], df_specific_fuel_consumption_curve['Specific fuel consumption (g/kWh)'])
plt.title('Specific fuel consumption curve from shop tests')
plt.xlabel('Pb [kW]')
plt.ylabel('Specific fuel consumption [g/kWh]')
plt.grid(True)
plt.show()

In [ ]:
# Brake power as a function of main engine fuel oil consumption
Pb_MFOC = {
    'Pb': np.array([0.25*9930, 0.35*9930, 0.5*9930, 0.716*9930, 0.75*9930, 9930]),
    'MFOC': np.array([175.9*0.25*9930/1000, 172.8*0.35*9930/1000, 167.7*0.5*9930/1000, 166*0.716*9930/1000, 167.7*0.75*9930/1000, 172.6*9930/1000])}
df_Pb_MFOC = pd.DataFrame(Pb_MFOC)

coefficients = np.polyfit(df_Pb_MFOC['MFOC'], df_Pb_MFOC['Pb'], 2)
a, b, c = coefficients

print(f" Pb = {a:.4f}x^2 + {b:.4f}x + {c:.4f}")


def function_Pb_MFOC(x):
    return a * x**2 + b * x + c
print(df_Pb_MFOC.head(6))


In [ ]:
# Plot of brake power versus fuel oil consumption

MFOC_fit = np.linspace(df_Pb_MFOC['MFOC'].min(), df_Pb_MFOC['MFOC'].max(), 100)
Pb_fit = function_Pb_MFOC(MFOC_fit)

plt.figure(figsize=(10, 6))
plt.scatter(df_Pb_MFOC['MFOC'], df_Pb_MFOC['Pb'], color = 'r', label='Shop-test data')
plt.plot(MFOC_fit, Pb_fit, label='Fitted polynomial')
plt.title('Pb-MFOC ')
plt.xlabel('MFOC [kg/hr]')
plt.ylabel('Pb [kW]')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Estimate brake power
df_q3 = df.copy()
FOC_t_day = df_q3['ME FOC [t/day]'] #[t/day]
MFOC = FOC_t_day*1000/24
Pb = function_Pb_MFOC(MFOC)
# Plot estimated versus measured brake power
plt.figure(figsize=(15, 6))
plt.scatter(df_q3['PROPELLER SHAFT POWER [kW]'], Pb, s=5, alpha=0.3)
plt.title('Estimated vs. measured brake power')
plt.xlabel('Measured brake power [kW]')
plt.ylabel('Estimated brake power [kW]')
plt.grid(True)
plt.show()

# Filter invalid measurements
mask = (
    (df_q3['ME FOC [t/day]'] > 0))
df_q3_filter = df_q3[mask].copy()
# Estimate brake power
FOC_t_day_2 = df_q3_filter['ME FOC [t/day]'] #[t/day]
MFOC_2 = FOC_t_day_2*1000/24

Pb_2 = function_Pb_MFOC(MFOC_2)
# Plot estimated versus measured brake power
plt.figure(figsize=(15, 6))
plt.scatter(df_q3_filter['PROPELLER SHAFT POWER [kW]'], Pb_2, s=5, alpha=0.3)
plt.title('Estimated vs. measured brake power')
plt.xlabel('Measured brake power [kW]')
plt.ylabel('Estimated brake power [kW]')
plt.grid(True)
plt.show()

# Question 4 — Outlier Detection

In [ ]:
rpm = df['PROPELLER REV [rpm]'].iloc[:1440].reset_index(drop=True)

window = 40
n = len(rpm)

z_scores = np.full(n, np.nan)
z_outliers = np.zeros(n, dtype=bool)
chauvenet_outliers = np.zeros(n, dtype=bool)
transient_flags = []

for i in range(0, n, window):
    segment = rpm.iloc[i:i+window]
    Nw = len(segment)

    mean = segment.mean()
    std = segment.std()

    if std == 0 or np.isnan(std):
        z = np.zeros(Nw)
        prob = np.ones(Nw)
    else:
        z = (segment - mean) / std
        prob = np.array([erfc(abs(val) / sqrt(2)) for val in z])

    z_scores[i:i+Nw] = z
    z_outliers[i:i+Nw] = np.abs(z) > 2
    chauvenet_outliers[i:i+Nw] = prob < (1 / (2 * Nw))

    transient_flags.append(std > 3)

print("Z-score outliers:", z_outliers.sum())
print("Chauvenet outliers:", chauvenet_outliers.sum())
print("Transient windows:", sum(transient_flags), "/", len(transient_flags))

# Plot
fig, ax1 = plt.subplots(figsize=(14,6))

ax1.plot(rpm.values, label='RPM')
ax1.scatter(np.where(z_outliers)[0], rpm[z_outliers], s=30, label='Z-score outliers')
ax1.scatter(np.where(chauvenet_outliers)[0], rpm[chauvenet_outliers], s=40, marker='x', label='Chauvenet outliers')

ax1.set_xlabel('Minute of Day 1')
ax1.set_ylabel('RPM')

ax2 = ax1.twinx()
ax2.plot(z_scores, alpha=0.4, label='Z-score')
ax2.set_ylabel('Z-score')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)

ax1.set_title('Outlier Detection in Propeller RPM (Day 1)')
ax1.grid(True, alpha=0.3)

plt.show()